In [ ]:
# HW1 Q3 - 5년 동안 행복지수가 한 번도 떨어지지 않은 나라
# 환경 준비: PySpark 설치, SparkContext 생성
!pip -q install pyspark
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

In [ ]:
# 데이터 내려받기 (textFile은 https 주소를 직접 못 읽으므로 파일로 받음, 내용은 손대지 않음)
!for y in 2015 2016 2017 2018 2019; do wget -q -O $y.csv "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/$y.csv"; done
# 컬럼 이름 확인 (PDF: 매해 컬럼이 조금씩 다르니 직접 출력해 확인)
!for y in 2015 2016 2017 2018 2019; do head -1 $y.csv; done

Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [ ]:
# 줄 -> (나라, 행복지수). 0번 칸 나라, 2번 칸 행복지수 (1번은 순위)
def parse(line):
    p = line.split(',')
    return (p[0], float(p[2]))                                  # -> ('Switzerland', 7.587)

# 읽기 -> 헤더 빼기 -> parse. 헤더 줄은 float('Happiness Score')에서 에러가 나므로 뺌
def load(path):
    rdd = sc.textFile(path)
    header = rdd.first()                                        # 헤더 줄 하나만 가져오는 action
    return rdd.filter(lambda line: line != header).map(parse)

s15 = load("2015.csv")
s16 = load("2016.csv")
s17 = load("2017.csv")
s18 = load("2018.csv")
s19 = load("2019.csv")

# join(다음 해) -> filter(안 떨어졌나) -> mapValues(최신 점수만 남기기, 키 안 바꾸니 mapValues)
# join 결과: ('나라', (작년 점수, 올해 점수))
j16 = s15.join(s16).filter(lambda t: t[1][0] <= t[1][1]).mapValues(lambda v: v[1])
j17 = j16.join(s17).filter(lambda t: t[1][0] <= t[1][1]).mapValues(lambda v: v[1])
j18 = j17.join(s18).filter(lambda t: t[1][0] <= t[1][1]).mapValues(lambda v: v[1])
j19 = j18.join(s19).filter(lambda t: t[1][0] <= t[1][1]).mapValues(lambda v: v[1])

In [ ]:
# 나라 이름만 꺼내(map) 이름순 정렬(sortBy)까지 Spark에서, collect가 마지막 action
result = j19.map(lambda t: t[0]).sortBy(lambda c: c).collect()
print(result)

['Benin', 'Burkina Faso', 'Cambodia', 'Cameroon', 'Chad', 'Congo (Brazzaville)', 'Czech Republic', 'Dominican Republic', 'Estonia', 'Finland', 'Gabon', 'Honduras', 'Hungary', 'Ivory Coast', 'Latvia', 'Malta', 'Mongolia', 'Niger', 'Philippines', 'Poland', 'Portugal', 'Romania', 'Senegal', 'Serbia', 'Slovakia', 'Syria', 'Tajikistan', 'Togo']
